# Fase 2 · Extracción automática de partidas desde XBRL (CMF)

**TFM — Índice de resiliencia financiera sectorial · Empresas del S&P IPSA · 2017–2025**

Este cuaderno lee los archivos **XBRL** de los estados financieros anuales descargados desde la CMF, extrae las partidas del núcleo definido en la sección 5.2 y las vuelca en la plantilla `Fase2_Base_Contable_Anual_IPSA.xlsx`, hoja **Captura_NoFin**.

**Qué hace**
- Lee archivos `.zip`, `.xbrl` o `.xml` tal como se descargan de la CMF.
- Extrae el año del documento y su columna comparativa, aplicando la regla de prelación: *el documento posterior prevalece*.
- Toma solo los valores consolidados totales, descartando desgloses por segmento o componente de patrimonio.
- Detecta la moneda de presentación (CLP o USD) y la registra en la hoja *Empresas*.
- Genera un informe de control: partidas faltantes, cuadre contable y años sin dato.
- Opcionalmente, carga los precios ajustados desde su CSV de Yahoo Finance.

**Qué no hace**
- **Bancos**: sus estados usan el Compendio bancario y se capturan a mano en *Captura_Fin* (4 bancos, 36 filas). Este cuaderno solo les carga los precios.
- **No inventa datos**: si una partida no está etiquetada con el elemento IFRS esperado, queda vacía y aparece en el informe para que usted la complete desde el PDF.

**Cómo usarlo**: ejecute las celdas en orden con `Mayús + Enter`. Solo necesita modificar la celda de **Parámetros**.

## Paso 0 · Preparar los archivos

1. Deje todos los `.zip` de **Estados financieros (XBRL)** descargados de la CMF dentro de una carpeta de Drive, por ejemplo `Mi unidad/TFM/XBRL`.
2. **No necesita renombrarlos ni descomprimirlos.** Pueden estar sueltos o en subcarpetas por año (`XBRL/2018/`, `XBRL/2020/`...): el cuaderno recorre todas las subcarpetas y **reconoce la empresa y el año leyendo el contenido de cada archivo**.
3. Suba también a Drive la plantilla `Fase2_Base_Contable_Anual_IPSA.xlsx`.

Si en el inventario (Paso 3B) algún archivo queda *sin asignar*, se corrige con una línea en el parámetro `ASIGNACION_MANUAL`.

## Paso 1 · Conectar Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('No está en Colab: se usarán rutas locales.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
for carpeta, subcarpetas, archivos in os.walk('/content/drive/MyDrive/TFM'):
    print(carpeta, '→', len(archivos), 'archivos')

/content/drive/MyDrive/TFM → 3 archivos
/content/drive/MyDrive/TFM/.ipynb_checkpoints → 0 archivos


In [ ]:
import os, glob
base = '/content/drive/MyDrive'
print('Drive conectado:', os.path.exists(base))
print('¿Existe la carpeta TFM?:', os.path.exists(base + '/TFM'))
if os.path.exists(base + '/TFM'):
    print('Contenido de TFM:', os.listdir(base + '/TFM'))

print('\nBuscando estados financieros en todo su Drive...')
hallados = [f for ext in ('zip', 'xbrl') for f in glob.glob(f'{base}/**/*.{ext}', recursive=True)]
print(len(hallados), 'archivos encontrados. Algunos ejemplos:')
for f in hallados[:8]:
    print('  ', f)

Drive conectado: True
¿Existe la carpeta TFM?: True
Contenido de TFM: ['informe_extraccion_xbrl_inventario.csv', 'Fase2_Base_Contable_Anual_IPSA.xlsx', 'informe_extraccion_xbrl.csv', '.ipynb_checkpoints']

Buscando estados financieros en todo su Drive...
738 archivos encontrados. Algunos ejemplos:
   /content/drive/MyDrive/TITULOS.zip
   /content/drive/MyDrive/Politicas de Vinculación con el medio unab.zip
   /content/drive/MyDrive/Rubric_ExportFile_AIND0202.202015.1465.TR_1598543140479.zip
   /content/drive/MyDrive/MARCELO CERDA.zip
   /content/drive/MyDrive/CARLOS BURGOS.zip
   /content/drive/MyDrive/KATHERINE FAJARDO.zip
   /content/drive/MyDrive/SEBASTIAN AGUAYO.zip
   /content/drive/MyDrive/VICTOR ACEVEDO.zip


## Paso 2 · Parámetros

**Esta es la única celda que debe editar.** Ajuste las rutas a las carpetas de su Drive.

`ESCALA`: el estándar XBRL registra los montos en unidades completas (pesos o dólares), mientras la plantilla trabaja en **miles**. Por eso se divide por 1.000. En el Paso 4 comprobará con el PDF que la escala es correcta.

In [ ]:
CARPETA_XBRL = '/content/drive/MyDrive/TFM/XBRL'
PLANTILLA    = '/content/drive/MyDrive/TFM/Fase2_Base_Contable_Anual_IPSA.xlsx'
SALIDA       = '/content/drive/MyDrive/TFM/Fase2_Base_Contable_Anual_IPSA_CARGADA.xlsx'
INFORME_CSV  = '/content/drive/MyDrive/TFM/informe_extraccion_xbrl.csv'

ESCALA = 1000            # unidades completas -> miles
ANIO_INI, ANIO_FIN = 2017, 2025
SOBRESCRIBIR = False     # False: no pisa celdas que usted ya haya llenado a mano

EMPRESA_PILOTO = 'AGUAS-A'   # empresa para la verificación del Paso 4

# Solo si el inventario deja archivos sin asignar: {'RUT sin dígito verificador': 'NEMOTECNICO'}
ASIGNACION_MANUAL = {
    # '61808000': 'AGUAS-A',
}

## Paso 3 · Definiciones y funciones de lectura

In [ ]:
import os, re, glob, zipfile
from collections import defaultdict
import pandas as pd
from lxml import etree

NS_XBRLI = "http://www.xbrl.org/2003/instance"
NS_XSI   = "http://www.w3.org/2001/XMLSchema-instance"

# Partida de la plantilla -> elementos de la taxonomía IFRS, en orden de preferencia
MAPEO = {
    "activo_corriente":                 ["CurrentAssets"],
    "pasivo_corriente":                 ["CurrentLiabilities"],
    "activo_total":                     ["Assets"],
    "pasivo_total":                     ["Liabilities"],
    "patrimonio_total":                 ["Equity"],
    "ingresos":                         ["Revenue", "RevenueFromContractsWithCustomers"],
    "costos_financieros":               ["FinanceCosts"],
    "ganancia_antes_impuestos":         ["ProfitLossBeforeTax"],
    "ganancia_ejercicio":               ["ProfitLoss"],
    "patrimonio_controladora":          ["EquityAttributableToOwnersOfParent"],
    "ganancia_controladora":            ["ProfitLossAttributableToOwnersOfParent"],
    "resultado_operacional_presentado": ["ProfitLossFromOperatingActivities"],
}
BALANCE = {"activo_corriente", "pasivo_corriente", "activo_total", "pasivo_total",
           "patrimonio_total", "patrimonio_controladora"}
NUCLEO = ["activo_corriente", "pasivo_corriente", "activo_total", "pasivo_total", "patrimonio_total",
          "ingresos", "costos_financieros", "ganancia_antes_impuestos", "ganancia_ejercicio"]

NO_FINANCIERAS = ["AGUAS-A", "COLBUN", "ENELAM", "ENELCHILE", "ECL", "ENTEL",
                  "CENCOSUD", "FALABELLA", "RIPLEY", "CCU", "ANDINA-B", "CONCHATORO",
                  "LTM", "PARAUCO", "CAP", "CMPC", "SQM-B", "COPEC"]
FINANCIERAS = ["CHILE", "BCI", "BSANTANDER", "ITAUCL"]


def parsear_nombre(ruta):
    """'AGUAS-A_2020.zip' -> ('AGUAS-A', 2020). Separa por el último guion bajo."""
    base = os.path.splitext(os.path.basename(ruta))[0]
    m = re.match(r"^(.+)_(\d{4})$", base.strip())
    return (m.group(1).upper(), int(m.group(2))) if m else None


def _es_instancia(raiz):
    return isinstance(raiz.tag, str) and raiz.tag == f"{{{NS_XBRLI}}}xbrl"


def leer_instancia(ruta):
    """Devuelve la raíz XML de la instancia XBRL, desde un .zip o un .xbrl/.xml."""
    parser = etree.XMLParser(huge_tree=True, remove_comments=True)
    if ruta.lower().endswith(".zip"):
        with zipfile.ZipFile(ruta) as z:
            for nombre in z.namelist():
                if not nombre.lower().endswith((".xbrl", ".xml")):
                    continue
                try:
                    raiz = etree.fromstring(z.read(nombre), parser)
                except etree.XMLSyntaxError:
                    continue
                if _es_instancia(raiz):
                    return raiz, nombre
        raise ValueError("el .zip no contiene una instancia XBRL")
    raiz = etree.parse(ruta, parser).getroot()
    if not _es_instancia(raiz):
        raise ValueError("el archivo no es una instancia XBRL")
    return raiz, os.path.basename(ruta)


def leer_contextos(raiz):
    """Contextos con su periodo; marca los que tienen dimensiones (segmentos, componentes)."""
    ctx = {}
    for c in raiz.iter(f"{{{NS_XBRLI}}}context"):
        per = c.find(f"{{{NS_XBRLI}}}period")
        if per is None:
            continue
        txt = lambda t: (per.findtext(f"{{{NS_XBRLI}}}{t}") or "").strip()[:10]
        ctx[c.get("id")] = {
            "dim": c.find(f".//{{{NS_XBRLI}}}segment") is not None
                   or c.find(f".//{{{NS_XBRLI}}}scenario") is not None,
            "instant": txt("instant"), "start": txt("startDate"), "end": txt("endDate"),
        }
    return ctx


def leer_unidades(raiz):
    uni = {}
    for u in raiz.iter(f"{{{NS_XBRLI}}}unit"):
        m = u.findtext(f"{{{NS_XBRLI}}}measure")
        if m:
            uni[u.get("id")] = m.strip().split(":")[-1].upper()
    return uni


def leer_hechos(raiz):
    """Hechos IFRS numéricos sin dimensiones: {(elemento, periodo): [(valor, moneda), ...]}"""
    ctx, uni = leer_contextos(raiz), leer_unidades(raiz)
    hechos = defaultdict(list)
    for el in raiz.iter():
        if not isinstance(el.tag, str):
            continue
        cref = el.get("contextRef")
        if cref is None:
            continue
        q = etree.QName(el)
        if "xbrl.ifrs.org" not in (q.namespace or ""):
            continue
        c = ctx.get(cref)
        if c is None or c["dim"] or el.get(f"{{{NS_XSI}}}nil") == "true":
            continue
        try:
            v = float((el.text or "").strip())
        except ValueError:
            continue
        per = ("I", c["instant"]) if c["instant"] else ("D", c["start"], c["end"])
        hechos[(q.localname, per)].append((v, uni.get(el.get("unitRef"), "")))
    return hechos


def extraer_anio(hechos, anio):
    """Partidas de un año. Devuelve (valores, elemento usado, monedas, alertas)."""
    p_bal = ("I", f"{anio}-12-31")
    p_res = ("D", f"{anio}-01-01", f"{anio}-12-31")
    val, orig, mon, alertas = {}, {}, set(), []
    for campo, elementos in MAPEO.items():
        per = p_bal if campo in BALANCE else p_res
        for e in elementos:
            h = hechos.get((e, per))
            if h:
                if len({v for v, _ in h}) > 1:
                    alertas.append(f"{campo}: {e} tiene valores distintos repetidos; se tomó el primero")
                val[campo], orig[campo] = h[0][0], e
                mon.update(m for _, m in h if m)
                break
    if "pasivo_total" not in val:          # respaldo: corriente + no corriente
        pc, pnc = hechos.get(("CurrentLiabilities", p_bal)), hechos.get(("NoncurrentLiabilities", p_bal))
        if pc and pnc:
            val["pasivo_total"] = pc[0][0] + pnc[0][0]
            orig["pasivo_total"] = "CurrentLiabilities + NoncurrentLiabilities"
    return val, orig, mon, alertas


import unicodedata

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"\s+", " ", t).strip()

# Palabra clave de la razón social -> nemotécnico. El ORDEN importa:
# las exclusiones van primero para que no se confundan con empresas del panel.
PALABRAS = [
    ("BANCO", None), ("CENCOSUD SHOPPING", None), ("AGUAS METROPOLITANAS", None),
    ("SUD AMERICANA DE VAPORES", None), ("PLAZA S", None), ("SMU", None),
    ("QUINENCO", None), ("SALFACORP", None), ("LA CONSTRUCCION", None),
    ("AGUAS ANDINAS", "AGUAS-A"), ("EMBOTELLADORA ANDINA", "ANDINA-B"),
    ("COLBUN", "COLBUN"), ("ENEL AMERICAS", "ENELAM"), ("ENEL CHILE", "ENELCHILE"),
    ("ENGIE ENERGIA", "ECL"), ("TELECOMUNICACIONES", "ENTEL"), ("CENCOSUD", "CENCOSUD"),
    ("FALABELLA", "FALABELLA"), ("RIPLEY", "RIPLEY"), ("CERVECERIAS UNIDAS", "CCU"),
    ("CONCHA Y TORO", "CONCHATORO"), ("LATAM", "LTM"), ("PARQUE ARAUCO", "PARAUCO"),
    ("CMPC", "CMPC"), ("QUIMICA Y MINERA", "SQM-B"), ("COPEC", "COPEC"),
    ("CAP S.A", "CAP"), ("CAP SA", "CAP"),
]


def rut_entidad(raiz):
    idt = raiz.find(f".//{{{NS_XBRLI}}}identifier")
    txt = (idt.text or "").strip() if idt is not None else ""
    return re.sub(r"\D", "", txt.split("-")[0])


def razon_social(raiz):
    """Busca el hecho de texto con el nombre de la entidad que informa."""
    for el in raiz.iter():
        if not isinstance(el.tag, str) or el.get("contextRef") is None:
            continue
        loc = etree.QName(el).localname.lower()
        if ("nameofreportingentity" in loc or "razonsocial" in loc or
                ("nombre" in loc and any(p in loc for p in ("entidad", "sociedad", "empresa")))):
            t = (el.text or "").strip()
            if t and not re.fullmatch(r"[\d.,\-\s]+", t):
                return t
    return ""


def anio_documento(raiz):
    """Año del ejercicio = fecha de balance más reciente; avisa si no es 31 de diciembre."""
    fechas = sorted({c["instant"] for c in leer_contextos(raiz).values() if c["instant"] and not c["dim"]})
    if not fechas:
        return None, "sin fechas de balance"
    ult = fechas[-1]
    return int(ult[:4]), ("" if ult.endswith("-12-31") else f"cierre {ult}: no es un estado anual")


def asignar(ruta, rut, nombre):
    if rut in ASIGNACION_MANUAL:
        return ASIGNACION_MANUAL[rut], "manual"
    p = parsear_nombre(ruta)
    if p and p[0] in NO_FINANCIERAS:
        return p[0], "nombre de archivo"
    n = normalizar(nombre)
    for clave, nemo in PALABRAS:
        if clave in n:
            return nemo, ("razón social" if nemo else "fuera del panel o banco")
    return None, "sin asignar"

print("Funciones listas.")

Funciones listas.


## Paso 3B · Inventario de archivos

Recorre la carpeta y todas sus subcarpetas, abre cada archivo y muestra **qué empresa y qué año** detectó. Revise el resumen:
- **Años por empresa**: cada empresa del panel debería tener sus documentos.
- **Archivos con observaciones**: los *sin asignar* se corrigen agregando su RUT en `ASIGNACION_MANUAL` (Paso 2) y volviendo a ejecutar desde allí. Los *fuera del panel o banco* se ignoran a propósito.

In [ ]:
rutas = [r for r in glob.glob(os.path.join(CARPETA_XBRL, "**", "*"), recursive=True)
         if os.path.isfile(r) and r.lower().endswith( (".zip", ".xbrl", ".xml"))]
filas, omitidos = [], 0
for ruta in rutas:
    rel = os.path.relpath(ruta, CARPETA_XBRL)
    try:
        raiz, _ = leer_instancia(ruta)
    except ValueError:
        omitidos += 1          # archivos auxiliares que no son instancias XBRL
        continue
    except Exception as ex:
        filas.append({"archivo": rel, "ruta": ruta, "estado": f"ERROR al leer: {ex}"})
        continue
    rut, nombre = rut_entidad(raiz), razon_social(raiz)
    anio, aviso = anio_documento(raiz)
    nemo, via = asignar(ruta, rut, nombre)
    filas.append({"archivo": rel, "ruta": ruta, "rut": rut, "razon_social": nombre, "anio": anio,
                  "nemotecnico": nemo, "asignado_por": via,
                  "estado": "OK" if nemo and anio and not aviso else (aviso or via)})

inventario = pd.DataFrame(filas)
validos = inventario[inventario.estado == "OK"] if len(inventario) else inventario
print(f"Archivos encontrados: {len(rutas)} · instancias XBRL: {len(inventario)} · "
      f"asignados correctamente: {len(validos)} · auxiliares omitidos: {omitidos}")

if len(validos):
    print("\nAÑOS DE DOCUMENTO POR EMPRESA")
    for n in NO_FINANCIERAS:
        a = sorted(int(x) for x in validos.loc[validos.nemotecnico == n, "anio"].unique())
        print(f"   {n:12s} {a if a else '— sin archivos —'}")

obs = inventario[inventario.estado != "OK"] if len(inventario) else inventario
if len(obs):
    print("\nARCHIVOS CON OBSERVACIONES")
    display(obs.drop(columns=["ruta"]))
inventario.drop(columns=["ruta"], errors='ignore').to_csv(INFORME_CSV.replace(".csv", "_inventario.csv"),
                                         index=False, encoding="utf-8-sig")

Archivos encontrados: 0 · instancias XBRL: 0 · asignados correctamente: 0 · auxiliares omitidos: 0


## Paso 4 · Verificación con la empresa piloto

Antes de procesar todo, revise **una empresa** contra su PDF. La tabla muestra los valores extraídos en **miles**.

Compare dos o tres cifras —el total de activos, los ingresos y la ganancia— con el estado financiero en PDF de la misma empresa y año:
- Si **coinciden**, la escala es correcta: continúe.
- Si le salen **1.000 veces más chicas**, cambie `ESCALA = 1` en el Paso 2 y vuelva a ejecutar desde allí.

In [ ]:
piloto = validos[validos.nemotecnico == EMPRESA_PILOTO] if len(validos) else validos
if piloto.empty:
    print(f"No hay archivos de {EMPRESA_PILOTO} en el inventario. Revise el Paso 3B o cambie EMPRESA_PILOTO.")
else:
    fila_p = piloto.sort_values("anio").iloc[-1]
    ruta, nemo, anio_doc = fila_p.ruta, fila_p.nemotecnico, int(fila_p.anio)
    raiz, interno = leer_instancia(ruta)
    h = leer_hechos(raiz)
    filas = {}
    for anio, col in [(anio_doc, "actual"), (anio_doc - 1, "comparativo")]:
        v, o, m, al = extraer_anio(h, anio)
        filas[f"{anio} ({col})"] = {k: v[k] / ESCALA if k in v else None for k in MAPEO}
        print(f"{anio}: moneda {', '.join(m) or 'no detectada'} · partidas del núcleo encontradas "
              f"{sum(k in v for k in NUCLEO)} de {len(NUCLEO)}")
        for a in al: print("  ⚠", a)
    print(f"\nDocumento: {os.path.basename(ruta)} · instancia: {interno}\n")
    pd.options.display.float_format = "{:,.0f}".format
    display(pd.DataFrame(filas))

No hay archivos de AGUAS-A en el inventario. Revise el Paso 3B o cambie EMPRESA_PILOTO.


### Si falta alguna partida del núcleo

Algunas empresas etiquetan una partida con un elemento distinto del esperado. Esta celda lista los elementos IFRS del documento piloto que contienen la palabra que usted indique, para identificar el correcto y agregarlo a `MAPEO` en el Paso 3.

In [ ]:
BUSCAR = "Revenue"   # cambie por: Liabilities, Finance, Profit, Equity...

if not piloto.empty:
    encontrados = sorted({(e, p) for (e, p) in h if BUSCAR.lower() in e.lower()})
    for e, p in encontrados:
        print(f"{e:55s} {p}  ->  {h[(e, p)][0][0] / ESCALA:,.0f}")

## Paso 5 · Extracción completa

Procesa todos los archivos en orden de ejercicio. Como el documento posterior prevalece, un año que aparece en dos documentos queda con la cifra del más reciente, que incorpora eventuales reexpresiones.

In [ ]:
registros, informe, errores = {}, [], []
monedas = defaultdict(set)

# Only proceed if validos is not empty
if not validos.empty:
    for _, fila in validos.sort_values("anio").iterrows():   # el documento posterior prevalece
        ruta, nemo, anio_doc = fila.ruta, fila.nemotecnico, int(fila.anio)
        if nemo not in NO_FINANCIERAS:
            continue
        try:
            raiz, _ = leer_instancia(ruta)
            h = leer_hechos(raiz)
        except Exception as ex:
            errores.append((os.path.basename(ruta), str(ex)))
            continue
        for anio, col in [(anio_doc, "actual"), (anio_doc - 1, "comparativo")]:
            if not ANIO_INI <= anio <= ANIO_FIN:
                continue
            v, o, m, al = extraer_anio(h, anio)
            if not v:
                continue
            monedas[nemo].update(m)
            rec = registros.setdefault((nemo, anio), {"nemotecnico": nemo, "anio": anio})
            for campo, valor in v.items():
                rec[campo] = valor / ESCALA
                informe.append({"nemotecnico": nemo, "anio": anio, "partida": campo,
                                "valor_miles": valor / ESCALA, "elemento_ifrs": o[campo],
                                "documento": fila.archivo, "columna": col})
            rec["fuente"] = f"XBRL {anio_doc}, {col}"
            for a in al:
                errores.append((os.path.basename(ruta), f"{anio} · {a}"))

datos = pd.DataFrame(sorted(registros.values(), key=lambda r: (NO_FINANCIERAS.index(r["nemotecnico"]), r["anio"])))
print(f"Observaciones empresa-año extraídas: {len(datos)} de {len(NO_FINANCIERAS) * (ANIO_FIN - ANIO_INI + 1)} esperadas")
print(f"Archivos con problemas: {len(errores)}")
for f, e in errores:
    print("  ⚠", f, "->", e)

Observaciones empresa-año extraídas: 0 de 162 esperadas
Archivos con problemas: 0


## Paso 6 · Informe de control

Tres revisiones antes de volcar a la plantilla:
1. **Años faltantes** por empresa: indican documentos no descargados o mal nombrados.
2. **Partidas del núcleo vacías**: debe completarlas desde el PDF.
3. **Cuadre contable**: activo total = pasivo total + patrimonio, con tolerancia de 0,5%.

In [ ]:
esperados = set(range(ANIO_INI, ANIO_FIN + 1))
print("1. AÑOS FALTANTES")
sin_archivos, parciales = [], []
for n in NO_FINANCIERAS:
    tiene = set(datos.loc[datos.nemotecnico == n, "anio"]) if len(datos) else set()
    falta = sorted(esperados - tiene)
    if len(falta) == len(esperados):
        sin_archivos.append(n)
    elif falta:
        parciales.append((n, falta))
for n, f in parciales:
    print(f"   {n:12s} faltan {f}")
if not parciales:
    print("   Ninguna empresa procesada tiene años incompletos.")
if sin_archivos:
    print(f"   Aún sin procesar ({len(sin_archivos)}): {', '.join(sin_archivos)}")

print("\n2. PARTIDAS DEL NÚCLEO VACÍAS")
vacias = []
for _, r in datos.iterrows():
    f = [c for c in NUCLEO if c not in r or pd.isna(r[c])]
    if f:
        vacias.append((r.nemotecnico, r.anio, f))
        print(f"   {r.nemotecnico:12s} {r.anio}: {', '.join(f)}")
if not vacias:
    print("   Ninguna.")

print("\n3. CUADRE CONTABLE (tolerancia 0,5%)")
descuadres = 0
for _, r in datos.iterrows():
    if all(c in r and pd.notna(r[c]) for c in ["activo_total", "pasivo_total", "patrimonio_total"]):
        dif = abs(r.activo_total - r.pasivo_total - r.patrimonio_total) / abs(r.activo_total)
        if dif > 0.005:
            descuadres += 1
            print(f"   {r.nemotecnico:12s} {r.anio}: diferencia {dif:.2%}")
if not descuadres:
    print("   Todas las observaciones cuadran.")

print("\nMONEDA DE PRESENTACIÓN DETECTADA")
for n in NO_FINANCIERAS:
    m = monedas.get(n)
    if m:
        print(f"   {n:12s} {', '.join(sorted(m))}" + ("   ⚠ más de una moneda" if len(m) > 1 else ""))

pd.DataFrame(informe).to_csv(INFORME_CSV, index=False, encoding="utf-8-sig")
print(f"\nTrazabilidad partida por partida guardada en: {INFORME_CSV}")

1. AÑOS FALTANTES
   Ninguna empresa procesada tiene años incompletos.
   Aún sin procesar (18): AGUAS-A, COLBUN, ENELAM, ENELCHILE, ECL, ENTEL, CENCOSUD, FALABELLA, RIPLEY, CCU, ANDINA-B, CONCHATORO, LTM, PARAUCO, CAP, CMPC, SQM-B, COPEC

2. PARTIDAS DEL NÚCLEO VACÍAS
   Ninguna.

3. CUADRE CONTABLE (tolerancia 0,5%)
   Todas las observaciones cuadran.

MONEDA DE PRESENTACIÓN DETECTADA

Trazabilidad partida por partida guardada en: /content/drive/MyDrive/TFM/informe_extraccion_xbrl.csv


## Paso 7 · Volcado a la plantilla Excel

Escribe los valores en las celdas amarillas de **Captura_NoFin** y la moneda en la hoja **Empresas**. Guarda un **archivo nuevo** (`..._CARGADA.xlsx`): su plantilla original no se modifica.

Con `SOBRESCRIBIR = False`, las celdas que usted ya haya llenado a mano se respetan. Las fórmulas de las hojas de ratios se recalculan al abrir el archivo en Excel.

In [ ]:
import openpyxl
import os

if not os.path.exists(PLANTILLA):
    print(f"Error: La plantilla Excel no se encontró en la ruta: {PLANTILLA}")
    print("Por favor, asegúrese de que el archivo 'Fase2_Base_Contable_Anual_IPSA.xlsx' exista en su Google Drive en la carpeta '/content/drive/MyDrive/TFM/'.")
else:
    wb = openpyxl.load_workbook(PLANTILLA)
    ws = wb["Captura_NoFin"]
    col = {ws.cell(1, j).value: j for j in range(1, ws.max_column + 1)}
    fila = {(ws.cell(r, 1).value, ws.cell(r, 4).value): r for r in range(2, ws.max_row + 1)}

    escritas = respetadas = 0
    for rec in registros.values():
        r = fila.get((rec["nemotecnico"], rec["anio"]))
        if r is None:
            continue
        for campo in list(MAPEO) + ["fuente"]:
            if campo not in rec or campo not in col:
                continue
            celda = ws.cell(r, col[campo])
            if celda.value not in (None, "") and not SOBRESCRIBIR:
                respetadas += 1
                continue
            celda.value = rec[campo]
            escritas += 1

    we = wb["Empresas"]
    for r in range(2, we.max_row + 1):
        n = we.cell(r, 2).value
        if n in monedas and len(monedas[n]) == 1 and we.cell(r, 8).value in (None, ""):
            we.cell(r, 8).value = next(iter(monedas[n]))

    wb.calculation.fullCalcOnLoad = True
    wb.save(SALIDA)
    print(f"Celdas escritas: {escritas} · celdas manuales respetadas: {respetadas}")
    print(f"Archivo guardado en: {SALIDA}")

Celdas escritas: 0 · celdas manuales respetadas: 0
Archivo guardado en: /content/drive/MyDrive/TFM/Fase2_Base_Contable_Anual_IPSA_CARGADA.xlsx


## Paso 8 (opcional) · Precios ajustados desde su CSV de Yahoo Finance

Carga el **precio ajustado de cierre** del último día hábil de cada año (2016–2025) en ambas hojas de captura, y el de 2016 en la hoja *Empresas*, que se usa para el retorno de 2017.

El CSV debe tener las **fechas en la primera columna** y **una columna por empresa**. Se reconocen nombres como `FALABELLA.SN` o `FALABELLA`. Use precios **ajustados** (*Adj Close*), no de cierre simple, para que el retorno incorpore los dividendos.

In [ ]:
PRECIOS_CSV = '/content/drive/MyDrive/TFM/precios_ipsa.csv'

# Nombres alternativos por cambio de razón social o de nemotécnico
ALIAS = {"LTM": ["LTM", "LAN"], "ITAUCL": ["ITAUCL", "ITAUCORP"]}

if not os.path.exists(PRECIOS_CSV):
    print("No se encontró el CSV de precios. Ajuste PRECIOS_CSV o omita este paso.")
else:
    px = pd.read_csv(PRECIOS_CSV, index_col=0)
    px.index = pd.to_datetime(px.index, errors="coerce", utc=True).tz_localize(None)
    px = px[px.index.notna()].sort_index()
    normal = {re.sub(r"\.SN$", "", str(c).strip().upper()): c for c in px.columns}

    cierres, sin_columna = {}, []
    for n in NO_FINANCIERAS + FINANCIERAS:
        c = next((normal[a] for a in ALIAS.get(n, [n]) if a in normal), None)
        if c is None:
            sin_columna.append(n)
            continue
        serie = pd.to_numeric(px[c], errors="coerce").dropna()
        for anio in range(ANIO_INI - 1, ANIO_FIN + 1):
            s = serie[serie.index.year == anio]
            if len(s):
                cierres[(n, anio)] = float(s.iloc[-1])

    wb = openpyxl.load_workbook(SALIDA if os.path.exists(SALIDA) else PLANTILLA)
    escritos = 0
    for hoja in ["Captura_NoFin", "Captura_Fin"]:
        w = wb[hoja]
        cp = [j for j in range(1, w.max_column + 1) if w.cell(1, j).value == "precio_ajustado_cierre"][0]
        for r in range(2, w.max_row + 1):
            k = (w.cell(r, 1).value, w.cell(r, 4).value)
            if k in cierres and (w.cell(r, cp).value in (None, "") or SOBRESCRIBIR):
                w.cell(r, cp).value = cierres[k]
                escritos += 1
    we = wb["Empresas"]
    for r in range(2, we.max_row + 1):
        k = (we.cell(r, 2).value, ANIO_INI - 1)
        if k in cierres and (we.cell(r, 9).value in (None, "") or SOBRESCRIBIR):
            we.cell(r, 9).value = cierres[k]
            escritos += 1
    wb.calculation.fullCalcOnLoad = True
    wb.save(SALIDA)
    print(f"Precios escritos: {escritos}")
    if sin_columna:
        print("Sin columna en el CSV (revise el nombre o agréguelo en ALIAS):", ", ".join(sin_columna))

No se encontró el CSV de precios. Ajuste PRECIOS_CSV o omita este paso.


## Paso 9 · Qué hacer después

1. Descargue `..._CARGADA.xlsx` desde su Drive y ábralo en **Excel** para que se recalculen las fórmulas.
2. Complete a mano, desde el PDF, las partidas que el Paso 6 marcó como vacías.
3. Capture los **4 bancos** en la hoja *Captura_Fin*.
4. Revise la hoja **Control**: cuando las 22 empresas digan *COMPLETA*, la Fase 2 está cerrada.

**Para su metodología (sección 5.2.2)**, puede declarar: *«La extracción de las partidas de las empresas no financieras se automatizó mediante un procedimiento en Python que lee las instancias XBRL publicadas por la CMF, selecciona los hechos consolidados sin dimensiones correspondientes a los elementos de la taxonomía IFRS definidos en la Tabla N°5 y registra, para cada valor, el elemento, el documento y la columna de origen. Los valores no etiquetados con el elemento esperado se completaron manualmente desde el estado financiero en formato PDF.»*